# Лабораторная работа №6

Многопоточная программа «Hello World!» в CUDA.

Напишите CUDA-программу, в которой создаётся 2 блока, содержащих
по 2 нити, и каждая нить выводит на экран строку «Hello World!».
Входные данные: нет.
Выходные данные: 4 строки «Hello World!».

In [1]:
%%writefile hello.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void cHello()  // Объявление функции-ядра
{
    printf("Hello World!\n");
}

int main()
{
    cHello<<<2, 2>>>(); // Запуск ядра. 2 блока по 2 нити = 4 нити суммарно, каждая выполнит cHello - независимо

    cudaError_t ce = cudaDeviceSynchronize(); // CPU ждёт пока GPU завершит все вычисления
    if (ce != cudaSuccess)
    {
        printf("Error: %s\n", cudaGetErrorString(ce));
    }

    return 0;
}

Writing hello.cu


In [2]:
!nvcc hello.cu -o hello && ./hello

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hello World!
Hello World!
Hello World!
Hello World!


## Ответы на вопросы:

### 1. Для каких устройств предназначен стандарт CUDA?

CUDA (Compute Unified Device Architecture) предназначен исключительно для графических ускорителей NVIDIA. Поддерживается начиная с серии G8x, включая линейки GeForce, Quadro и Tesla.

### 2. Какую модель параллельного программирования реализует этот стандарт?

CUDA реализует модель SIMT (Single Instruction, Multiple Threads) одна инструкция выполняется одновременно множеством нитей. Это разновидность модели GPGPU (General-Purpose computing on GPU) использование видеокарты для вычислений общего назначения, не связанных с графикой.

### 3. Как организуется программа в стандарте CUDA?

Программа состоит из двух частей:

- Host-код - выполняется на CPU, обычный C/C++. Управляет памятью и запускает ядра.
- Device-код (ядра) - выполняется на GPU, помечается квалификатором __global__.

Нити организованы иерархически:

- Грид (Grid) - совокупность всех блоков
- Блок (Block) - группа нитей, имеющих общую память
- Нить (Thread) - минимальная единица выполнения

В нашей лабе: грид из 2 блоков × 2 нити = 4 нити суммарно.

### 4. Какую роль играет команда kernel, каковы её параметры?

Команда запуска ядра служит для вызова функции-ядра на GPU. Синтаксис:

kernel<<<Dg, Db, Ns, S>>>(params)

Параметр Dg имеет тип dim3 и задаёт размер грида — то есть количество блоков.

Параметр Db также имеет тип dim3 и задаёт размер каждого блока — количество нитей в нём.

Параметр Ns типа size_t является опциональным и указывает размер динамически выделяемой общей памяти.

Параметр S типа cudaStream_t тоже опциональный и определяет поток, в котором запускается ядро.

В нашем примере cHello<<<2, 2>>>() означает запуск ядра в 2 блоках по 2 нити в каждом, итого 4 нити.